# AU Case Study - Phase-Diagram Placement Check (Read-Only)

> Data source: `014_phase_placement_check.py`; theoretical reference = the UK's
> `025_exp_r26_condition_phase_diagram.py` conditional proposition (derived in log space):
> **multiplicative correction + renormalization reduces error iff rho(r, c) > sigma_c/(2*sigma_r)**,
> i.e. above the theory line alpha\* = sigma_ratio / 2 on the phase diagram is the help
> region, and below it is the hurt region.
> This notebook is **read-only display only, and writes no artifacts**; all numbers come
> from saved files under `data/processed/evaluation/` (nothing is hand-typed).
>
> **Background**: multiplicative correction on AU's GNN base **significantly improves**
> accuracy (GNNpostNP vs GNN: Delta=-12.23, Holm p=0.006) -- the opposite of the antagonistic
> direction seen in the UK (Delta=+1.93, worse). This step checks whether this "non-replication"
> is **predicted** by the conditional proposition -- i.e. whether AU's GNN x multiplicative
> combination systematically falls on the help side of the theory line, forming "two sides
> of the same theory line" together with the UK's hurt-side placement, rather than
> indicating the theory has failed.
>
> **Convention** (identical, word for word, to the UK's equivalent script): station-level
> rho = corr(log F, log rho_resid), sigma_r = std(log rho_resid), sigma_c = std(log F)
> (ddof=0); epsilon = regional total demand x 1e-6; delta RMSE uses the raw allocation
> value (actual_col='peak_mw'). Combination universe = {Uniform, GPM, GNN x 3 seeds} x
> 12 SA4 x {N, P, NP} = 180 points; 240 built-in anchor checks (static-arm tolerance 1e-9 /
> GNN-arm 4-decimal-place, half-ulp tolerance of 5e-5).

In [ ]:
%matplotlib inline
# Environment and artifact paths (read-only)
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 160)

AU_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVAL = AU_DIR / "data" / "processed" / "evaluation"

au = pd.read_csv(EVAL / "au_phase_points.csv", encoding="utf-8-sig")
merged = pd.read_csv(EVAL / "uk_au_phase_points_merged.csv", encoding="utf-8-sig")
summary = json.loads((EVAL / "phase_placement_summary.json").read_text(encoding="utf-8"))
uk = merged[merged["country"] == "UK"].copy()

print(f"AU placements: {len(au)} rows | Merged: {len(merged)} rows (UK {len(uk)} + AU {len(au)})")
print(f"Anchors: {summary['anchor_check']['n_anchors']} checks, all passed = "
      f"{summary['anchor_check']['all_passed']} (static max deviation "
      f"{summary['anchor_check']['max_abs_dev_static_anchors']:.1e} / GNN "
      f"{summary['anchor_check']['max_abs_dev_gnn_anchors']:.1e})")

## 1. Core Test Readout (Conclusion Field Generated by Rule: value = min(predicted fraction, actual fraction))

Decision thresholds are the same as in the UK's equivalent analysis: value >= 0.9 -> supported;
>= 0.5 -> partially_supported.

In [ ]:
for key in ["au_gnn_mult_in_help_region", "au_gnn_np_in_help_region",
            "au_static_in_help_region", "same_theory_line_two_sides"]:
    blk = summary[key]
    print("=" * 100)
    print(f"[{key}] {blk['description']}")
    for k, v in blk.items():
        if k in ("description", "by_signal"):
            continue
        print(f"  {k} = {v:.4f}" if isinstance(v, float) else f"  {k} = {v}")
print("=" * 100)
print("AU GNN x multiplicative, by signal:")
for sig, cell in summary["au_gnn_mult_in_help_region"]["by_signal"].items():
    print(f"  {sig:3s}: predicted help {cell['frac_predicted_help']:.3f} / actual help "
          f"{cell['frac_actual_help']:.3f} | median (rho, sigma_c/sigma_r) = "
          f"({cell['median_pearson_log']:.3f}, {cell['median_sigma_ratio']:.3f})")
mv = summary["uk_contrast"]["au_vs_uk_gnn_medians"]
print(f"\nGNN-base median coordinate comparison: AU ({mv['au_median_pearson_log']:.3f}, "
      f"{mv['au_median_sigma_ratio']:.3f}) vs UK ({mv['uk_median_pearson_log']:.3f}, "
      f"{mv['uk_median_sigma_ratio']:.3f})")

## 2. Confusion Matrix (Theoretical Prediction help/hurt x Actual DeltaRMSE Sign)

In [ ]:
def cm_row(name, cm):
    return {"Group": name, "pred help/actual help": cm["pred_help_actual_help"],
            "pred help/actual hurt": cm["pred_help_actual_hurt"],
            "pred hurt/actual help": cm["pred_hurt_actual_help"],
            "pred hurt/actual hurt": cm["pred_hurt_actual_hurt"],
            "total": cm["total"], "accuracy": round(cm["accuracy"], 4)}

rows = [cm_row("AU all", summary["confusion_matrix"])]
rows += [cm_row(f"AU {bt}", cm)
         for bt, cm in summary["confusion_by_base_type"].items()]
display(pd.DataFrame(rows).set_index("Group"))
print("UK reference (transcribed from the UK's equivalent analysis): GNN x NP predicted hurt "
      f"{summary['uk_contrast']['uk_gnn_np']['frac_predicted_hurt']:.3f} / actual hurt "
      f"{summary['uk_contrast']['uk_gnn_np']['frac_actual_hurt']:.3f}")

## 3. Combined UK + AU Phase Diagram (Draft for the Paper's Three-Case Figure)

X-axis: sigma_c/sigma_r; Y-axis: rho = corr(log F, log rho_resid); black dashed line =
theory line alpha\* = sigma_ratio/2 (above = help, below = hurt). Filled markers = actual
improvement (DeltaRMSE<0), open markers = actual worsening. Once Germany's training
finishes, a third country can be added to the same figure.

In [ ]:
COLORS = {"uniform": "#7f7f7f", "gpm": "#1f77b4", "gnn": "#d62728"}
LABELS = {"uniform": "Uniform base", "gpm": "GPM base", "gnn": "GNN base"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
for ax, (name, df) in zip(axes, [("UK (16 regions, 240 points)", uk),
                                 ("AU (12 regions, 180 points)", au)]):
    xs = np.linspace(0, max(1.6, df["sigma_ratio"].max() * 1.05), 100)
    ax.plot(xs, xs / 2, "k--", lw=1.2, label="Theory line ρ = σ_ratio/2")
    ax.fill_between(xs, xs / 2, 1.05, color="#2ca02c", alpha=0.06)
    ax.text(0.02, 0.98, "help region", transform=ax.transAxes, va="top",
            fontsize=10, color="#2ca02c")
    ax.text(0.98, 0.02, "hurt region", transform=ax.transAxes, va="bottom",
            ha="right", fontsize=10, color="#8c564b")
    for bt in ("uniform", "gpm", "gnn"):
        sub = df[df["base_type"] == bt]
        helped = sub[sub["actual_help"]]
        hurt = sub[~sub["actual_help"]]
        ax.scatter(helped["sigma_ratio"], helped["pearson_log"], s=22,
                   color=COLORS[bt], alpha=0.75, label=f"{LABELS[bt]} (actual improvement)")
        ax.scatter(hurt["sigma_ratio"], hurt["pearson_log"], s=26,
                   facecolors="none", edgecolors=COLORS[bt], alpha=0.9,
                   label=f"{LABELS[bt]} (actual worsening)")
    ax.set_title(name)
    ax.set_xlabel(r"$\sigma_c/\sigma_r$")
    ax.grid(alpha=0.25)
axes[0].set_ylabel(r"$\rho = \mathrm{corr}(\log F,\ \log\rho_{resid})$")
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, labels, fontsize=7.5, loc="lower right", framealpha=0.9)
axes[0].set_ylim(-0.35, 1.05)
fig.suptitle("UK + AU phase-diagram placements: same theory line, two cases on opposite sides", y=1.02)
fig.tight_layout()
plt.show()

## 4. GNN x NP Single-Arm Focus (the Exact Dual of the UK's Arm)

The UK analysis's proposition for this arm was "GNN x NP falls in the hurt region"
(value=0.583, partially_supported); AU's mirror proposition is "GNN x NP falls in the
help region." The relative position of these two point sets on the same theory line is
direct graphical evidence for the "two sides of the same theory line" narrative.

In [ ]:
uk_np = uk[(uk["base_type"] == "gnn") & (uk["signal"] == "NP")]
au_np = au[(au["base_type"] == "gnn") & (au["signal"] == "NP")]

fig, ax = plt.subplots(figsize=(7.5, 6))
xs = np.linspace(0, 1.7, 100)
ax.plot(xs, xs / 2, "k--", lw=1.4, label="Theory line ρ = σ_ratio/2")
ax.fill_between(xs, xs / 2, 1.05, color="#2ca02c", alpha=0.06)
ax.scatter(uk_np["sigma_ratio"], uk_np["pearson_log"], s=40, marker="s",
           color="#9467bd", alpha=0.8, label=f"UK GNN x NP ({len(uk_np)} points)")
ax.scatter(au_np["sigma_ratio"], au_np["pearson_log"], s=40, marker="o",
           color="#d62728", alpha=0.8, label=f"AU GNN x NP ({len(au_np)} points)")
for df, c in ((uk_np, "#9467bd"), (au_np, "#d62728")):
    ax.scatter([df["sigma_ratio"].median()], [df["pearson_log"].median()],
               s=260, marker="*", color=c, edgecolors="k", zorder=5,
               linewidths=0.8)
ax.annotate(f"UK median ({uk_np['sigma_ratio'].median():.2f}, "
            f"{uk_np['pearson_log'].median():.2f})",
            (uk_np["sigma_ratio"].median(), uk_np["pearson_log"].median()),
            textcoords="offset points", xytext=(10, -18), fontsize=9)
ax.annotate(f"AU median ({au_np['sigma_ratio'].median():.2f}, "
            f"{au_np['pearson_log'].median():.2f})",
            (au_np["sigma_ratio"].median(), au_np["pearson_log"].median()),
            textcoords="offset points", xytext=(10, 8), fontsize=9)
ax.text(0.03, 0.97, "help region", transform=ax.transAxes, va="top",
        fontsize=11, color="#2ca02c")
ax.text(0.97, 0.03, "hurt region", transform=ax.transAxes, va="bottom",
        ha="right", fontsize=11, color="#8c564b")
ax.set_xlabel(r"$\sigma_c/\sigma_r$")
ax.set_ylabel(r"$\rho = \mathrm{corr}(\log F,\ \log\rho_{resid})$")
ax.set_title("GNN x NP: UK and AU fall on opposite sides of the same theory line")
ax.set_ylim(-0.35, 1.05)
ax.set_xlim(0, 1.7)
ax.grid(alpha=0.25)
ax.legend(fontsize=9, loc="lower right")
fig.tight_layout()
plt.show()

print(f"AU GNN x NP: predicted help {au_np['predicted_help'].mean():.3f} / "
      f"actual help {au_np['actual_help'].mean():.3f}")
print(f"UK GNN x NP: predicted hurt {(~uk_np['predicted_help']).mean():.3f} / "
      f"actual hurt {(~uk_np['actual_help']).mean():.3f}")
ts = summary["same_theory_line_two_sides"]
print(f"Two sides of the same theory line, value = {ts['value']:.3f} -> {ts['verdict']}"
      f"(rule: {ts['value_rule']})")

## 5. Conclusion (all numbers sourced from phase_placement_summary.json, nothing hand-typed)

- **AU's "non-replication" is predicted by the conditional proposition**: AU's GNN x
  multiplicative combination systematically falls in the theoretical help region (see
  the all-signal value in Section 1; for the NP arm alone, 36/36 points are predicted
  help and all are actually improved), while the UK's same arm falls on the hurt side --
  the two cases are **two-sided placements on the same theory line
  rho = sigma_c/(2*sigma_r)**, not a failure of the theory.
- **A coordinate-level explanation for the placement difference**: the alignment between
  AU's GNN-base residual and the correction signal (median rho ~= 0.78) is far higher
  than the UK's (~= 0.30), and sigma_c/sigma_r is slightly lower (0.43 vs 0.52) -- rho
  far exceeds the threshold sigma_ratio/2, so multiplicative correction improves accuracy
  rather than being antagonistic.
- **Upper bound on the strength of the two-sided narrative**: the merged verdict is capped
  by the UK side's value of 0.583 (the UK analysis's original value for this proposition) --
  the AU side's value is 1.0, and the shortfall is in the purity of the UK's hurt side,
  consistent with the UK analysis's own partially_supported conclusion.
- Once Germany's training finishes, a third country can be appended to
  `uk_au_phase_points_merged.csv` and this notebook's phase diagram reused directly.